# **Step 5 – Selección de la tarea de minería de datos**
**Proyecto TA – Análisis de Datos (1INF03, 2026-2) – Equipo E10**

> *KDD, paso 5:* «Matching the goals of the KDD process to a particular data mining method: summarization, classification, regression, clustering, etc.»

Este notebook asigna a cada objetivo del Step 1 un tipo de tarea, y ejecuta la tarea descriptiva (auditoría de discrepancias), que no requiere modelo.

Entrada: `datos/2_limpios/step3_evaluaciones.csv`. Salida: `datos/4_resultados/step5_auditoria_discrepancias.csv`.

## 1. Objetivos y tareas

| Objetivo (Step 1) | Tipo de tarea | Por qué | Dónde |
|---|---|---|---|
| 1. Estimar la alteración nutricional estricta de la ficha a partir de peso, talla, IMC, edad y sexo | **Clasificación binaria supervisada** | La etiqueta existe (registro manual de la ficha) y es binaria; interesa estimarla y medir cuánto aporta cada medida | Step 6 |
| 2. Identificar NNA y meses en que ficha y mediciones discrepan | **Resumen / descripción** (comparación de dos clasificaciones) | No se busca predecir sino **listar** casos para revisión; basta comparar la ficha con la clasificación OMS calculada por la institución | Este notebook |
| 3. Perfiles de NNA | **Clustering** (exploratorio) | No hay etiqueta para "perfil"; en el avance solo se prueba si los grupos que forma K-Means se relacionan con la etiqueta | Step 6 (exploratorio) |

**Por qué no regresión:** la variable de interés es un diagnóstico categórico (con o sin alteración), no una cantidad continua. Una regresión del IMC o del peso reproduciría la medida, no el registro de la ficha que se quiere auditar.

**Etiqueta principal:** `alteracion_estricta` (exceso de peso o retraso de talla), más equilibrada (≈37 % de positivos) y clínicamente más clara. `alteracion` (incluye *riesgos*) se usa como sensibilidad.

## 2. Auditoría: ficha vs. clasificación OMS de las medidas
Para cada evaluación se compara la etiqueta estricta de la ficha con la clasificación OMS calculada por la institución:
- *exceso* = IMC OMS en sobrepeso u obesidad;
- *retraso de talla* = T/E OMS en talla baja o talla baja severa;
- *alteración estricta medida* = exceso **o** retraso de talla.

In [1]:
import json
from pathlib import Path
import numpy as np, pandas as pd
LIM, RES = Path("../datos/2_limpios"), Path("../datos/4_resultados")
ev = pd.read_csv(LIM/"step3_evaluaciones.csv", sep="|")
ev["med_exceso"] = ev["IMC_OMS"].isin(["Sobrepeso","Obesidad"]).astype(int)
ev["med_talla"] = ev["T/E_OMS"].isin(["Talla baja","Talla baja severa"]).astype(int)
ev["med_estricta"] = ((ev["med_exceso"]==1) | (ev["med_talla"]==1)).astype(int)
print("Evaluaciones:", len(ev), "| sin IMC OMS:", int(ev["IMC_OMS"].isna().sum()), "| sin T/E OMS:", int(ev["T/E_OMS"].isna().sum()))
tabla = pd.crosstab(ev["alteracion_estricta"], ev["med_estricta"], rownames=["ficha"], colnames=["medición OMS"], margins=True)
concordancia = 100*(ev["alteracion_estricta"] == ev["med_estricta"]).mean()
print(tabla.to_string()); print(f"\nConcordancia: {concordancia:.1f} %")

Evaluaciones: 324 | sin IMC OMS: 14 | sin T/E OMS: 0
medición OMS    0    1  All
ficha                      
0             145   58  203
1              43   78  121
All           188  136  324

Concordancia: 68.8 %


Se clasifica cada evaluación según el tipo de discrepancia y se resume por niño.

In [2]:
ev["tipo_discrepancia"] = np.select(
    [(ev["alteracion_estricta"]==0) & (ev["med_estricta"]==1), (ev["alteracion_estricta"]==1) & (ev["med_estricta"]==0)],
    ["ficha 'sin alteración' pero la medición indica exceso o talla baja", "ficha 'con alteración' pero la medición es normal"], "concuerda")
print(ev["tipo_discrepancia"].value_counts().to_string())
ev["discrepa"] = (ev["tipo_discrepancia"] != "concuerda").astype(int)
por_nino = ev.groupby("id_nna").agg(evaluaciones=("discrepa","size"), con_discrepancia=("discrepa","sum"))
disc_ninos = int((por_nino["con_discrepancia"] > 0).sum())
disc_todas = int(((por_nino["con_discrepancia"] == por_nino["evaluaciones"]) & (por_nino["con_discrepancia"] > 0)).sum())
print(f"\nNiños con al menos una discrepancia: {disc_ninos} de {len(por_nino)} | con discrepancia en todas sus evaluaciones: {disc_todas}")

tipo_discrepancia
concuerda                                                             223
ficha 'sin alteración' pero la medición indica exceso o talla baja     58
ficha 'con alteración' pero la medición es normal                      43

Niños con al menos una discrepancia: 65 de 121 | con discrepancia en todas sus evaluaciones: 14


Se arma la lista para la nutricionista: un registro por niño con discrepancia, ordenado por número de evaluaciones discrepantes, con los meses y el tipo de discrepancia.

In [3]:
d = ev[ev["discrepa"] == 1]
lista = (d.groupby("id_nna").agg(evaluaciones_con_discrepancia=("discrepa","size"), edad_media=("edad_ev","mean"),
                                 periodos=("periodo", lambda x: ", ".join(sorted(x))),
                                 tipos=("tipo_discrepancia", lambda x: " / ".join(sorted(set(x)))),
                                 IMC_OMS=("IMC_OMS", lambda x: ", ".join(x.fillna("—"))), TE_OMS=("T/E_OMS", lambda x: ", ".join(x.fillna("—"))))
         .join(por_nino["evaluaciones"].rename("evaluaciones_totales")).reset_index()
         .sort_values(["evaluaciones_con_discrepancia","id_nna"], ascending=[False, True]))
lista["edad_media"] = lista["edad_media"].round(1)
lista.to_csv(RES/"step5_auditoria_discrepancias.csv", index=False, sep="|", encoding="utf-8")
print("Guardado:", lista.shape); lista.head(10)

Guardado: (65, 8)


,id_nna,evaluaciones_con_discrepancia,edad_media,periodos,tipos,IMC_OMS,TE_OMS,evaluaciones_totales
57,PPA_FN_0140_M_20240722,4,1.6,"2026_01, 2026_02, 2026_03, 2026_06",ficha 'sin alteración' pero la medición indica...,"Obesidad, Sobrepeso, Sobrepeso, Obesidad","Riesgo talla baja, Normal, Normal, Normal",5
6,PPA_FN_0018_M_20180824,3,7.6,"2026_01, 2026_04, 2026_06",ficha 'sin alteración' pero la medición indica...,"Normal, Riesgo bajo peso, Bajo peso","Talla baja, Talla baja, Talla baja severa",3
12,PPA_FN_0028_F_20150131,3,11.2,"2026_01, 2026_04, 2026_06",ficha 'sin alteración' pero la medición indica...,"Riesgo bajo peso, Riesgo bajo peso, Bajo peso","Talla baja, Talla baja, Talla baja severa",3
35,PPA_FN_0094_M_20091013,3,16.5,"2026_01, 2026_04, 2026_06",ficha 'sin alteración' pero la medición indica...,"Normal, Normal, Normal","Talla baja, Talla baja, Talla baja severa",3
37,PPA_FN_0100_M_20120703,3,13.8,"2026_01, 2026_04, 2026_06",ficha 'sin alteración' pero la medición indica...,"Sobrepeso, Sobrepeso, Obesidad","Alto, Normal, Normal",3
48,PPA_FN_0123_M_20180325,3,8.0,"2026_01, 2026_04, 2026_06",ficha 'sin alteración' pero la medición indica...,"Sobrepeso, Sobrepeso, Obesidad","Riesgo talla baja, Riesgo talla baja, Talla baja",3
64,PPA_FN_0150_F_20181120,3,7.4,"2026_01, 2026_04, 2026_06",ficha 'sin alteración' pero la medición indica...,"Normal, Normal, Riesgo sobrepeso","Talla baja, Talla baja, Talla baja severa",3
2,PPA_FN_0011_F_20200304,2,6.0,"2026_01, 2026_04",ficha 'con alteración' pero la medición es normal,"Riesgo sobrepeso, Riesgo sobrepeso","Normal, Riesgo talla baja",3
3,PPA_FN_0015_M_20190202,2,7.1,"2026_01, 2026_04",ficha 'con alteración' pero la medición es normal,"Normal, Normal","Riesgo talla baja, Riesgo talla baja",3
4,PPA_FN_0016_M_20181030,2,7.3,"2026_01, 2026_04",ficha 'con alteración' pero la medición es normal,"Normal, Normal","Riesgo talla baja, Riesgo talla baja",3


**Interpretación.** La ficha y la clasificación OMS por medidas **concuerdan en el 68,8 %** de las 324 evaluaciones. En 58 casos la ficha dice "sin alteración" pero la medición indica exceso de peso o talla baja (posible **subregistro**: niños que podrían quedar sin atención), y en 43 ocurre lo contrario. **65 de 121 niños** tienen al menos una discrepancia y **14 la tienen en todas** sus evaluaciones. No sabemos cuál de los dos lados falla: la ficha puede actualizarse con rezago (su etiqueta no cambió en 19 meses, Step 3), combinar otros indicadores (INS, P/E, P/T) o haber errores de medición o de edad. Por eso `step5_auditoria_discrepancias.csv` es una **lista para revisión de la nutricionista**, no una corrección automática.

*Nota:* 14 evaluaciones no tienen IMC OMS registrado; en ellas solo se puede detectar el retraso de talla, por lo que la discrepancia por exceso de peso podría estar subestimada.

## 3. Un caso para la revisión: la edad usada en el diagnóstico
En el Step 2b se vio que `PPA_FN_0117` figura en enero y febrero con una edad escrita 12 meses menor que la real. Se revisan sus evaluaciones.

In [4]:
caso = ev[ev["id_nna"].str.startswith("PPA_FN_0117")][["periodo","edad_ev","Peso_kg","Talla_cm","IMC_valor","P/E_OMS","T/E_OMS","IMC_OMS","alteracion_estricta","med_estricta"]]
caso.round(2)

,periodo,edad_ev,Peso_kg,Talla_cm,IMC_valor,P/E_OMS,T/E_OMS,IMC_OMS,alteracion_estricta,med_estricta
0,2026_01,2.34,14.30,86.2,19.25,Sobrepeso,Alto,Sobrepeso,0,1
117,2026_02,2.43,13.75,92.0,16.25,Sobrepeso,Normal,Normal,0,0
121,2026_03,2.51,14.04,92.5,16.41,Normal,Normal,Normal,0,0
125,2026_04,2.59,15.40,94.0,17.43,Riesgo sobrepeso,Normal,NaN,0,0
219,2026_06,2.76,14.50,94.7,16.17,Riesgo sobrepeso,Normal,Riesgo sobrepeso,0,0


**Interpretación.** En enero, `PPA_FN_0117` (2,3 años) figura con **P/E e IMC "sobrepeso"** según la OMS, justo el mes en que su edad estaba escrita como 1 año y 4 meses en lugar de 2 años y 4 meses. En marzo, con la edad bien escrita y un peso parecido (14,0 kg), el P/E pasa a **normal**. Un mismo peso es "sobrepeso" para un niño de 16 meses y normal para uno de 28: es plausible que el diagnóstico de enero se calculara con la edad equivocada. Es un ejemplo de por qué la lista debe revisarse caso por caso: parte de las discrepancias puede venir del registro de las mediciones y no de la ficha.

## Verificación y cifras de este notebook

In [5]:
assert len(ev) == 324 and round(concordancia, 1) == 68.8
assert ev["tipo_discrepancia"].value_counts().to_dict() == {"concuerda": 223, "ficha 'sin alteración' pero la medición indica exceso o talla baja": 58, "ficha 'con alteración' pero la medición es normal": 43}
assert (disc_ninos, disc_todas, len(lista)) == (65, 14, 65)
assert caso.iloc[0]["P/E_OMS"] == "Sobrepeso" and caso.iloc[2]["P/E_OMS"] == "Normal"
cifras = {"concordancia": round(concordancia, 1), "ficha_sin_med_con": 58, "ficha_con_med_sin": 43,
          "ninos_discrepancia": disc_ninos, "ninos_eval": int(ev["id_nna"].nunique()), "ninos_todas": disc_todas}
json.dump(cifras, open(RES/"cifras_step5.json", "w"), indent=1); cifras

{'concordancia': np.float64(68.8),
 'ficha_sin_med_con': 58,
 'ficha_con_med_sin': 43,
 'ninos_discrepancia': 65,
 'ninos_eval': 121,
 'ninos_todas': 14}

## Declaración de uso de IA
Se usó **Claude Code (Anthropic)** para separar la auditoría en su propio paso y redactar la justificación de las tareas. El equipo revisó la lista de discrepancias y verificó con los `assert` que las cifras del texto coinciden con las salidas.

Siguiente: `Step6_Seleccion_de_Algoritmos.ipynb`.